# Estimating Arrhenius parameters from isothermal experiments

The thermal decomposition of benzenediazonium chloride in water,

$$\mathrm{C_6H_5N_2Cl}+\mathrm{H_2O}\;\longrightarrow\;\mathrm{C_6H_5OH}+\mathrm{N_2}+\mathrm{HCl},$$

is a classical first-order reaction: the nitrogen it releases makes the conversion
easy to follow. In a batch experiment at the temperature $T$ the diazonium
concentration decays as

$$\frac{dc}{dt}=-k(T)\,c,\qquad k(T)=k_\mathrm{ref}\exp\!\Big[-\frac{E_a}{R}\Big(\frac1T-\frac1{T_\mathrm{ref}}\Big)\Big].$$

Writing the Arrhenius law around a reference temperature $T_\mathrm{ref}$ in the middle
of the experiments decouples the two parameters $k_\mathrm{ref}$ and $E_a$, which makes
the estimation well conditioned.

OneSimulation solves each experiment as a homogeneous reaction network
(`ReactionKinetics`, classical Runge-Kutta). The `SimulationStudyService` wraps the five
simulations into one study: for a trial pair $(\ln k_\mathrm{ref},E_a)$ it builds and runs
all five, measures the mean squared deviation from the $N$ data points

$$\mathrm{MSD}=\frac1N\sum_{\mathrm{experiments}}\sum_{j}\big(c_\mathrm{sim}(t_j)-c_\mathrm{data}(t_j)\big)^2,$$

and its projected-gradient optimizer (finite-difference gradients from real runs,
backtracking line search, bounds on both parameters) minimises it. The optimizer
steps along the plain gradient, so both parameters are given on a scale of order one:
$\ln k_\mathrm{ref}$ and $E_a$ in units of 100 kJ/mol.

**Reference.** The data are synthetic: they are generated by OneSimulation itself from
chosen "true" values, $k_\mathrm{ref}=1.0\times10^{-3}$ 1/s at 50 °C and
$E_a=115$ kJ/mol, of the order reported for this reaction, with normally distributed
noise of standard deviation $s=0.05$ mol/m³ (0.5 % of the initial 10 mol/m³). A
correct estimation must recover the true values within the uncertainty that this
noise allows and leave residuals of the size of the noise, $\mathrm{MSD}\approx s^2$ (Y. Bard, *Nonlinear Parameter Estimation*, Academic Press 1974; on the
reaction: E. A. Moelwyn-Hughes, *The Kinetics of Reactions in Solution*, Oxford 1947).

In [ ]:
import numpy as np

from onesim.cad.geometry import LumpedDomain
from onesim.meshing import LumpedMesh
from onesim.numerics import Numerics
from onesim.numerics.discretization import LumpedParameters
from onesim.numerics.time import ClassicalRungeKutta4
from onesim.physics.chemistry import ReactionIR, ReactionKinetics, ReactionNetworkIR, SpeciesIR
from onesim.simulation import Problem, Simulation
from onesim.simulation.initial_conditions import InitialValue
from onesim.simulation.studies import SimulationStudyService, StudyParameter
from onesim.visualization import theme, plots

theme.set()

## Experiments and model

Five isothermal experiments from 40 °C to 60 °C, one hour each, with twelve samples
every five minutes. `experiments(values)` builds the five simulations for one
parameter pair; it is the model the study varies.

In [ ]:
gas, reference_temperature, initial, noise = 8.314462618, 323.15, 10.0, 0.05
temperatures = (313.15, 318.15, 323.15, 328.15, 333.15)
samples = np.linspace(300.0, 3600.0, 12)
true = {"ln_k_ref": np.log(1.0e-3), "activation_energy": 1.15}
cell = LumpedDomain()
experiments = lambda values: {T: ReactionKinetics(domain=cell.system, temperature=T, network=ReactionNetworkIR(
    "diazonium_decomposition", "1",
    species=(SpeciesIR("diazonium", 0, 1e-27, composition={"H": 1}), SpeciesIR("products", 0, 1e-27, composition={"H": 1})),
    reactions=(ReactionIR("decomposition", {"diazonium": 1}, {"products": 1},
                          {"kind": "arrhenius", "activation_energy": 1e5 * values["activation_energy"], "boltzmann_constant": gas,
                           "pre_exponential": np.exp(values["ln_k_ref"] + 1e5 * values["activation_energy"] / (gas * reference_temperature))}),),
    strict_balance=True)) for T in temperatures}
run = lambda models: {T: np.interp(samples, *Simulation(
    problem=Problem(name=f"Batch decomposition at {T - 273.15:.0f} °C", geometry=cell, physics=(model,),
                    initial_conditions=(InitialValue(model.concentration("diazonium"), initial), InitialValue(model.concentration("products"), 0.0))),
    mesh=LumpedMesh(cell), numerics=Numerics(spatial=LumpedParameters(), time_integrator=ClassicalRungeKutta4())).run(
    start_time=0.0, end_time=3600.0, time_step=20.0).field(model.concentration("diazonium")).time_series()) for T, model in models.items()}

## Synthetic measurements

The true parameters are run once and normally distributed noise is added (fixed
seed, so the notebook is reproducible).

In [ ]:
rng = np.random.default_rng(2026)
clean = run(experiments(true))
data = {T: c + rng.normal(0.0, noise, samples.size) for T, c in clean.items()}
print(f"{len(temperatures)} experiments x {samples.size} samples = {len(temperatures) * samples.size} data points, noise s = {noise} mol/m³")

## Estimation

Bounds: $k_\mathrm{ref}$ between $10^{-4}$ and $10^{-2}$ 1/s, $E_a$ between 60 and 180
kJ/mol; finite-difference steps of $10^{-6}$ of each interval. The optimizer starts far
from the truth, at $k_\mathrm{ref}=3\times10^{-4}$ 1/s and $E_a=80$ kJ/mol, and stops when
the projected gradient of the MSD falls below $10^{-4}$ (mol/m³)² per unit parameter or
the line search finds no further decrease.

In [ ]:
study = SimulationStudyService(
    (StudyParameter("ln_k_ref", np.log(1e-4), np.log(1e-2)), StudyParameter("activation_energy", 0.6, 1.8)),
    experiments,
    lambda computed: {"msd": sum(float(np.sum((computed[T] - data[T]) ** 2)) for T in temperatures) / (len(temperatures) * samples.size)},
    solve=run)
best = study.optimize(("msd",), start={"ln_k_ref": np.log(3e-4), "activation_energy": 0.8}, step=1e-6, tolerance=1e-4, maximum_iterations=60, maximum_runs=800)
fitted = run(experiments(best.parameters))

## Comparison with the true values

The parameter uncertainty follows from the curvature of $\chi^2$ at the optimum
(Gauss-Newton: covariance $=(J^TJ)^{-1}$ with the sensitivities $J$ of the weighted
residuals); the sensitivities are taken from two more runs with shifted parameters.

In [ ]:
shifted = {name: run(experiments(dict(best.parameters) | {name: best.parameters[name] + step}))
           for name, step in (("ln_k_ref", 1e-4), ("activation_energy", 1e-5))}
sensitivity = np.column_stack([np.concatenate([(shifted[name][T] - fitted[T]) / (step * noise) for T in temperatures])
                               for name, step in (("ln_k_ref", 1e-4), ("activation_energy", 1e-5))])
sigma = np.sqrt(np.diag(np.linalg.inv(sensitivity.T @ sensitivity)))
print(f"optimizer: {best.iterations} iterations, {best.run_count} study runs ({5 * best.run_count} kinetic simulations), "
      f"χ² = N·MSD/s² = {len(temperatures) * samples.size * best.objectives['msd'] / noise**2:.1f} for {len(temperatures) * samples.size - 2} degrees of freedom")
print(f"k_ref (50 °C): estimated {np.exp(best.parameters['ln_k_ref']):.5e} 1/s ± {100 * sigma[0]:.2f} %, true {np.exp(true['ln_k_ref']):.5e} 1/s "
      f"(deviation {100 * (np.exp(best.parameters['ln_k_ref'] - true['ln_k_ref']) - 1):+.2f} %)")
print(f"E_a: estimated {100 * best.parameters['activation_energy']:.2f} ± {100 * sigma[1]:.2f} kJ/mol, true {100 * true['activation_energy']:.2f} kJ/mol "
      f"(deviation {(best.parameters['activation_energy'] - true['activation_energy']) / sigma[1]:+.2f} standard deviations)")
print(f"rms residual {np.sqrt(np.mean(np.concatenate([fitted[T] - data[T] for T in temperatures]) ** 2)):.4f} mol/m³ (noise {noise})")
excess = sum(float(np.sum(((clean[T] - data[T]) / noise) ** 2)) for T in temperatures) - len(temperatures) * samples.size * best.objectives["msd"] / noise**2
print(f"χ² at the true values exceeds the minimum by Δχ² = {excess:.1f} (joint 95 % region of two parameters: Δχ² < 5.99, 99 %: < 9.21)")

With 60 noisy samples the estimate scatters around the truth by the printed standard
deviations; for this noise realisation the true pair lies just outside the joint 95 %
confidence region and inside the 99 % region, as happens for one data set in fifty. A
different seed gives a different, equally valid set of data and estimates.

## Figure

The five experiments with the fitted model; every parameter pair the optimizer ran
(gradient probes and line-search trials included); and the Arrhenius plot of the rate constants.

In [ ]:
path = np.array([[100 * r.parameters["activation_energy"], np.exp(r.parameters["ln_k_ref"])] for r in best.runs]
                + [[100 * best.parameters["activation_energy"], np.exp(best.parameters["ln_k_ref"])]])
inverse = np.linspace(1 / 335.0, 1 / 311.0, 50)
fig = plots.figure(1, 3, size=(17, 5))
plots.line(points={f"{T - 273.15:.0f} °C": (samples / 60, data[T]) for T in temperatures},
           reference={f"{T - 273.15:.0f} °C": (samples / 60, fitted[T]) for T in temperatures},
           xlabel="time [min]", ylabel="diazonium concentration [mol/m³]", title="Data and fitted model (dashed)",
           legend="upper right", ax=fig[0])
plots.line(points={"trial points": (path[:, 0], 1e3 * path[:, 1]), "start": (path[:1, 0], 1e3 * path[:1, 1]),
                   "true values": (np.array([115.0]), np.array([1.0])), "estimate": (path[-1:, 0], 1e3 * path[-1:, 1])},
           scale="logy", xlabel="activation energy E_a [kJ/mol]", ylabel="k_ref at 50 °C [1/ks]",
           title="All trial points of the optimizer", legend="upper right", ax=fig[1])
plots.line(points={"fitted model": (1e3 / np.array(temperatures), np.log(np.exp(best.parameters["ln_k_ref"]) * np.exp(
               -1e5 * best.parameters["activation_energy"] / gas * (1 / np.array(temperatures) - 1 / reference_temperature))))},
           reference={"true Arrhenius law": (1e3 * inverse, true["ln_k_ref"] - 1e5 * true["activation_energy"] / gas * (inverse - 1 / reference_temperature))},
           xlabel="1000 / T [1/K]", ylabel="ln k [k in 1/s]", title="Arrhenius plot", legend="upper right", ax=fig[2])
plots.show()

## Going further

Halve the noise and the standard deviations halve; drop the 40 °C and 60 °C experiments
and watch $E_a$ become much less certain while $k_\mathrm{ref}$ hardly changes, the
reason for spreading experiments over a wide temperature range. Real data, for example
nitrogen volumes read from a gas burette, replace the synthetic `data` dictionary
directly; the measured volume is proportional to the converted amount
$c_0-c(t)$.